In [8]:
import random
import numpy as np

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score

print("----- CYBER ATTACK DETECTION USING GENETIC ALGORITHM -----")
print("Name: HEMALATHA S USN: 1BM25CS454")
print()

X, y = make_classification(
    n_samples=1000,
    n_features=12,
    n_informative=8,
    n_redundant=2,
    random_state=42
)

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.4, random_state=42, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5,
    random_state=42, stratify=y_temp
)

POP_SIZE = 10
GENERATIONS = 10
MUTATION_RATE = 0.1
N_FEATURES = X.shape[1]

def create_individual():
    chromosome = [random.randint(0, 1)
                  for _ in range(N_FEATURES)]
    if sum(chromosome) == 0:
        chromosome[random.randrange(N_FEATURES)] = 1
    return chromosome

def fitness(chromosome):
    selected = np.where(np.array(chromosome) == 1)[0]

    model = RandomForestClassifier(
        n_estimators=30, random_state=42
    )
    model.fit(X_train[:, selected], y_train)

    predictions = model.predict(X_val[:, selected])
    score = f1_score(y_val, predictions)

    penalty = 0.01 * len(selected) / N_FEATURES
    return score - penalty

random.seed(42)
population = [create_individual() for _ in range(POP_SIZE)]

for generation in range(GENERATIONS):

    population.sort(key=fitness, reverse=True)

    parents = population[:POP_SIZE // 2]
    offspring = [parents[0][:]]

    while len(offspring) < POP_SIZE:
        p1, p2 = random.sample(parents, 2)

        point = random.randint(1, N_FEATURES - 1)
        child = p1[:point] + p2[point:]

        for i in range(N_FEATURES):
            if random.random() < MUTATION_RATE:
                child[i] = 1 - child[i]

        if sum(child) == 0:
            child[random.randrange(N_FEATURES)] = 1

        offspring.append(child)

    population = offspring
    best = max(population, key=fitness)
    print("Generation", generation + 1,
          "Fitness:", round(fitness(best), 4))

best = max(population, key=fitness)
selected = np.where(np.array(best) == 1)[0]

X_combined = np.vstack((X_train, X_val))
y_combined = np.concatenate((y_train, y_val))

model = RandomForestClassifier(
    n_estimators=100, random_state=42
)
model.fit(X_combined[:, selected], y_combined)

predictions = model.predict(X_test[:, selected])

print("\nBest Feature Selection:", best)
print("Selected Feature Indices:", selected.tolist())
print("Test Accuracy:",
      round(accuracy_score(y_test, predictions) * 100, 2), "%")
print("Test F1 Score:",
      round(f1_score(y_test, predictions), 4))
print("Confusion Matrix:")
print(confusion_matrix(y_test, predictions))
print("\n0 = Normal Traffic, 1 = Cyber Attack")


----- CYBER ATTACK DETECTION USING GENETIC ALGORITHM -----
Name: HEMALATHA S USN: 1BM25CS454

Generation 1 Fitness: 0.8411
Generation 2 Fitness: 0.8448
Generation 3 Fitness: 0.8571
Generation 4 Fitness: 0.8571
Generation 5 Fitness: 0.8571
Generation 6 Fitness: 0.8571
Generation 7 Fitness: 0.8571
Generation 8 Fitness: 0.8571
Generation 9 Fitness: 0.8625
Generation 10 Fitness: 0.8697

Best Feature Selection: [0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1]
Selected Feature Indices: [1, 3, 4, 5, 6, 7, 8, 9, 10, 11]
Test Accuracy: 86.5 %
Test F1 Score: 0.8657
Confusion Matrix:
[[86 14]
 [13 87]]

0 = Normal Traffic, 1 = Cyber Attack
